# Assignment 1: Model

**Author:** Lino Kaiser.

A thermal zone is a space represented by one homogeneous air temperature. The project building has two rooms and two air temperatures, $T_1(t)$ and $T_2(t)$. Heat passes between them through the shared wall and the closed internal door.    

The setting is inspired by the teaching and office use of ZHAW's Campus Technikumstrasse in Winterthur. The campus location is real. The pavilion, dimensions, orientation, materials, schedules and technical systems below are explicitly chosen modelling assumptions, not real ZHAW building data.

This definition is the baseline for the project. Geometry, materials, openings and operating assumptions can be found below.

## 1. Layout and dimensions

![Two-zone floor plan](figures/building_layout.svg)

| Quantity | Zone 1: project room | Zone 2: office |
|---|---:|---:|
| Reference dimensions | 6.0 × 6.0 m | 3.0 × 6.0 m |
| Floor and roof area | 36.0 m² | 18.0 m² |
| Model height | 3.0 m | 3.0 m |
| Air volume | 108.0 m³ | 54.0 m³ |
| Occupants | 4 students | 2 staff |
| Windows | Two south-facing | One north-facing |

The total reference plan is 9.0 × 6.0 m: 54.0 m² floor area and 162.0 m³ air volume. All dimensions use an **idealized thermal reference grid**. Wall thickness is omitted from the footprint and volumes, but included in thermal resistances and capacities.

 The larger room represents group work and the smaller room an office. Different window orientations create a useful contrast in solar gains. Different occupant counts give different absolute heat inputs. The office is reached through the project room. The baseline assumes both doors closed.

## 2. Envelope, openings and materials

Layers below run from the room towards the outside, ground or other room. All constructions are generic assumptions.

| Element | Chosen construction |
|---|---|
| External walls | 15 mm gypsum plaster + 200 mm brick + 120 mm mineral wool + 15 mm render |
| Flat roof | 15 mm plaster + 150 mm concrete + 160 mm mineral wool |
| Ground floor | 20 mm timber finish + 150 mm concrete slab + 100 mm EPS |
| Shared partition | 12.5 mm plaster + 100 mm brick + 12.5 mm plaster |
| D1: external door | Insulated opaque door, 1.00 × 2.10 m; whole-door U = 1.5 W/(m² K) |
| D2: internal door | 40 mm solid timber, 0.90 × 2.10 m |
| W1 and W2 | Each 2.00 × 1.50 m, south-facing; sill 0.90 m |
| W3 | 1.50 × 1.20 m, north-facing; sill 1.00 m |

We assume double glazing: **whole-window** U = 1.3 W/(m² K), glazing fraction 0.80, **glazing-only** solar factor g = 0.60. Transmitted solar input is opening area × 0.80 × 0.60 × facade irradiance. The frame is included through the whole-window U-value and glazing fraction, not a separate thermal node. Window and exterior-door thermal storage are neglected initially.

Material values are rounded assumed inputs.

| Material | Conductivity λ [W/(m K)] | Density ρ [kg/m³] | Specific heat c [J/(kg K)] |
|---|---:|---:|---:|
| Brick | 0.70 | 1800 | 840 |
| Mineral wool | 0.040 | 120 | 1030 |
| Gypsum plaster | 0.25 | 900 | 1000 |
| Render | 0.80 | 1800 | 840 |
| Concrete | 1.70 | 2300 | 880 |
| EPS | 0.035 | 20 | 1450 |
| Timber | 0.13 | 600 | 1600 |

For the preliminary heat-loss estimates, we use **combined** inside and outside surface resistances of 0.13 and 0.04 m² K/W. 

Horizontal conduction within the roof and slab across the zone boundary is neglected.

## 3. Occupancy and internal heat gains

We assume **Monday to Friday, 08:00 to 18:00**. People, equipment and lighting are constant during this interval and zero outside it.

| Occupied-period input | Project room | Office |
| --- | ---: | ---: |
| People, at 75 W sensible heat each | 300 W | 150 W |
| Laptops, at 50 W each | 200 W | 100 W |
| Lighting, at 5 W/m² | 180 W | 90 W |
| **Total internal sensible gains** | **680 W** | **340 W** |

All internal sensible gains enter the zone air balance initially, avoiding additional radiant/convective split parameters. Windows remain closed. Effective outdoor air exchange is **1.0 air changes per hour (ACH)** when occupied and **0.3 ACH** otherwise, without heat recovery. These are model assumptions, not a ventilation-compliance assessment. At 1.0 ACH, the flows are 108 and 54 m³/h, or 27 m³/h per occupant in either room.

The closed internal door has no additional interzone airflow. Its conduction is retained. No extra door-opening or window-opening ventilation is added to the prescribed air exchange.

## 4. Location and boundary conditions

| Boundary/input | Assumption |
| --- | --- |
| Location | Hypothetical detached teaching annex inspired by ZHAW Campus Technikumstrasse, Winterthur |
| Orientation | Project windows south, office window north; east/west walls opaque |
| Outdoor air | Common prescribed temperature for both zones; all exterior walls and the roof exposed |
| Weather for the later simulation | Selected Zurich Airport EPW, WMO 066700, TMYx 2009-2023 |
| Ground | Constant 10 °C below the insulated slab |
| Exterior shading | No neighbouring-building or tree shadows in the baseline |
| Solar properties | Wall absorptance 0.60; roof 0.80; ground reflectance 0.20 |
| Heating | Separate heat input per zone; target 21 °C occupied, 16 °C otherwise |
| Cooling | None in the baseline |
| Initial temperatures | All model temperature states start at 20 °C |
| First steady-state check | Outdoor 0 °C; ground 10 °C; no sun or internal gains; explicitly prescribed heating inputs |

Both rooms have their own air-temperature output. Heat crosses the partition and internal door according to the temperature difference.

## 5. Parameters and geometry checks

The following cells define reusable inputs and calculate areas, occupied-period gains and preliminary U-values. All opening dimensions are width × height.

In [1]:
from building import definition

building = definition['building']
zones = definition['zones']
openings = definition['openings']
materials = definition['materials']
assemblies = definition['assemblies']
boundary = definition['boundary']

In [2]:
import math

surface_areas = {}
for name, z in zones.items():
    floor = z["width_m"] * z["depth_m"]
    volume = floor * z["height_m"]
    # Each zone has north and south facades, plus one 6 m side facade.
    gross_exterior = (2 * z["width_m"] + z["depth_m"]) * z["height_m"]
    own_openings = [o for o in openings if o["zone"] == name]
    window_area = sum(o["width_m"] * o["height_m"] for o in own_openings
                      if o["kind"] == "window")
    door_area = sum(o["width_m"] * o["height_m"] for o in own_openings
                    if o["kind"] == "external_door")
    opaque_exterior = gross_exterior - window_area - door_area
    gains = (z["people"] * boundary["person_sensible_W"]
             + z["laptops"] * boundary["laptop_W"]
             + floor * boundary["lighting_W_m2"])
    surface_areas[name] = {
        "floor_and_roof_m2": floor, "volume_m3": volume,
        "gross_exterior_wall_m2": gross_exterior,
        "windows_m2": window_area, "external_door_m2": door_area,
        "opaque_exterior_wall_m2": opaque_exterior,
        "occupied_internal_gains_W": gains,
    }
    print(f"{name}: floor/roof {floor:.2f} m²; volume {volume:.2f} m³; "
          f"opaque exterior walls {opaque_exterior:.2f} m²; "
          f"windows {window_area:.2f} m²; external door {door_area:.2f} m²; "
          f"internal gains {gains:.0f} W")

shared_gross = building["reference_depth_m"] * building["height_m"]
internal_door = next(o for o in openings if o["kind"] == "internal_door")
internal_door_area = internal_door["width_m"] * internal_door["height_m"]
shared_opaque = shared_gross - internal_door_area
print(f"Shared partition: {shared_opaque:.2f} m² opaque + "
      f"{internal_door_area:.2f} m² internal door = {shared_gross:.2f} m² gross")

assert math.isclose(sum(a["floor_and_roof_m2"] for a in surface_areas.values()), 54.0)
assert math.isclose(sum(a["volume_m3"] for a in surface_areas.values()), 162.0)
assert math.isclose(sum(a["opaque_exterior_wall_m2"]
                        + a["windows_m2"] + a["external_door_m2"]
                        for a in surface_areas.values()), 90.0)
assert math.isclose(shared_opaque + internal_door_area, 18.0)
assert all(a["opaque_exterior_wall_m2"] > 0 for a in surface_areas.values())
assert all(m["lambda_W_mK"] > 0 and m["rho_kg_m3"] > 0 and m["cp_J_kgK"] > 0
           for m in materials.values())

print("\nPreliminary one-dimensional U estimates [W/(m² K)]:")
r_inside = boundary["inside_combined_film_R_m2K_W"]
r_outside = boundary["outside_combined_film_R_m2K_W"]
for name, layers in assemblies.items():
    r_layers = sum(thickness / materials[material]["lambda_W_mK"]
                   for material, thickness in layers)
    r_films = (2 * r_inside if name in ["partition", "internal_door"]
               else r_inside if name == "floor"
               else r_inside + r_outside)
    u_value = 1 / (r_layers + r_films)
    print(f"  {name}: {u_value:.3f}")
print("Geometry and positive-property checks passed.")

project_room: floor/roof 36.00 m²; volume 108.00 m³; opaque exterior walls 45.90 m²; windows 6.00 m²; external door 2.10 m²; internal gains 680 W
office: floor/roof 18.00 m²; volume 54.00 m³; opaque exterior walls 34.20 m²; windows 1.80 m²; external door 0.00 m²; internal gains 340 W
Shared partition: 16.11 m² opaque + 1.89 m² internal door = 18.00 m² gross

Preliminary one-dimensional U estimates [W/(m² K)]:
  external_wall: 0.283
  roof: 0.232
  floor: 0.310
  partition: 1.989
  internal_door: 1.762
Geometry and positive-property checks passed.


## 6. Model assumptions

- Each room has one homogeneous air temperature and its own air heat capacity.
- Each room's external walls are combined into one wall element. Roof and floor are separate elements. Each solid element has one storage node at the centre of its masonry/concrete layer; the heat capacities of its other layers are lumped into this node.
- A massless inner-surface node separates solid conduction from surface heat exchange. Each inner surface has convection and linearized radiation in parallel. The room air temperature is used as a proxy for mean radiant temperature.
- The inside film resistance remains 0.13 m² K/W. We split its coefficient into assumed convection h_c = 3.0 W/(m² K) and radiation h_r = 1/0.13 - 3.0. Their sum preserves the chosen film resistance. Exterior surface exchange remains included in the outside film resistance of 0.04 m² K/W.
- The shared partition has one storage node. The closed internal door, windows and exterior door are resistance-only paths. Door/window openings are subtracted from the wall areas.
- Air exchange is prescribed by ACH (Air Changes per Hour). Internal heat gains enter the air nodes. All validation cases have zero solar input.
- A proportional-controller branch follows the supplied tutorial. Its gain is zero in free-running cases. The heated validation case uses 500 W/K per room and requires positive heating power in both rooms. Heating saturation and switching are outside these steady-state tests.

This gives **15 temperature nodes: 9 storage states and 6 massless surfaces**. Numbering starts at zero. The full layer resistances are retained, while the distribution of solid thermal storage is deliberately simplified.

In [3]:
import numpy as np
import pandas as pd
from IPython.display import display

import dm4bem
from building import definition

building = definition["building"]
zones = definition["zones"]
openings = definition["openings"]
materials = definition["materials"]
assemblies = definition["assemblies"]
boundary = definition["boundary"]

zone_names = ["project_room", "office"]
air_nodes = [0, 7]
rho_air = 1.2       # kg/m³, assumed
cp_air = 1005       # J/(kg K), assumed
r_inside = boundary["inside_combined_film_R_m2K_W"]
r_outside = boundary["outside_combined_film_R_m2K_W"]
h_convection = 3.0
h_radiation = 1 / r_inside - h_convection

## 7. Conductances and capacities

For a layer of thickness d and area S:

$$R_\mathrm{layer}=\frac{d}{\lambda S},\qquad C_\mathrm{layer}=\rho c d S.$$

A surface branch has G = hS. Outdoor-air exchange has

$$G_\mathrm{vent}=\rho_\mathrm{air}c_\mathrm{air}\frac{\mathrm{ACH}\,V}{3600}.$$

For each envelope element, the masonry/concrete layer is split into two half-resistances around its storage node. The other layer resistances remain on the appropriate side of that node. The inside film is represented by the two parallel surface branches.

Geometry is calculated first, followed by the branch conductances [W/K] and node heat capacities [J/K].

In [4]:
from thermal_model import areas, door_area, partition_area, element_properties

display(pd.DataFrame(areas).T.rename(columns={
    "external_wall": "Opaque walls [m²]", "roof": "Roof [m²]",
    "floor": "Floor [m²]", "windows": "Windows [m²]",
    "external_door": "Exterior door [m²]", "volume": "Air volume [m³]",
}))

,Opaque walls [m²],Roof [m²],Floor [m²],Windows [m²],Exterior door [m²],Air volume [m³]
project_room,45.9,36.0,36.0,6.0,2.1,108.0
office,34.2,18.0,18.0,1.8,0.0,54.0


In [5]:
from thermal_model import (
    node_names, capacities, branches, ventilation_branches, heating_branches,
    partition_branches, internal_door_branch,
)

branch_table = pd.DataFrame(
    branches, columns=["Path", "From", "To node", "G [W/K]"]
)
branch_table.index.name = "Branch"
node_table = pd.DataFrame({"Temperature node": node_names, "C [J/K]": capacities})
node_table.index.name = "Node"
display(branch_table.round(3))
display(node_table.round(1))

,Path,From,To node,G [W/K]
Branch,,,,
0,project_room: external_wall outside,outdoor,1,14.337
1,project_room: external_wall inside,1,2,226.268
2,project_room: external_wall convection,2,0,137.700
3,project_room: external_wall radiation,2,0,215.377
4,project_room: roof outside,outdoor,3,8.815
5,project_room: roof inside,3,4,345.763
6,project_room: roof convection,4,0,108.000
7,project_room: roof radiation,4,0,168.923
8,project_room: floor outside,ground,5,12.408


,Temperature node,C [J/K]
Node,,
0,Project air,130248.0
1,Project wall mass,16221610.8
2,Project wall surface,0.0
3,Project roof mass,12127536.0
4,Project roof surface,0.0
5,Project floor mass,11725200.0
6,Project floor surface,0.0
7,Office air,65124.0
8,Office wall mass,12086690.4


## 8. Numbered thermal circuit

![Numbered thermal circuit](figures/thermal_circuit.svg)

Arrows define positive branch flow; a negative result means heat flows in the opposite direction. The branch and node numbers correspond to the conductance and capacity tables in section 7.

The split radiation branch uses the zone air temperature as the mean-radiant-temperature approximation. There is no additional inside film resistance in series with these branches.

## 9. DAE matrices and input pointers

The matrix convention as seen in the supplied material is:

$$C\dot{\theta}=-A^TGA\theta+A^TG b_T+f_Q,$$
$$q=G(-A\theta+b_T).$$

A row of $A$ has −1 at its starting unknown node and +1 at its ending node; a prescribed-temperature source has only the ending-node entry. $G=\mathrm{diag}(g_0,\ldots,g_{33})$ and $C=\mathrm{diag}(C_0,\ldots,C_{14})$.

The matrices and pointer vectors are constructed in [thermal_model.py](thermal_model.py). $G$ and $C$ are diagonal matrices containing the conductances and capacities listed above. The vectors `b` and `f` mark where temperature and heat-flow inputs enter; `y` selects the two room-air outputs.


In [6]:
from thermal_model import A, base_conductance, b, G, C, f, y, n_nodes, n_branches

state_nodes = np.flatnonzero(capacities)
input_names = (
    [f"b_T[{j}]: {branches[j][1]}" for j in np.flatnonzero(b)]
    + [f"f_Q[{j}]: {node_names[j]}" for j in np.flatnonzero(f)]
)

print(f"DAE: A {A.shape}, G {G.shape}, C {C.shape}")
print("State temperature nodes:", state_nodes)
print("b =", b)
print("f =", f)
print("y =", y)
display(pd.DataFrame({"Input": input_names}))
display(pd.DataFrame(A.astype(int)).rename_axis(index="Branch", columns="Node"))


DAE: A (34, 15), G (34, 34), C (15, 15)
State temperature nodes: [ 0  1  3  5  7  8 10 12 14]
b = [1 0 0 0 1 0 0 0 1 0 0 0 1 1 1 1 1 0 0 0 1 0 0 0 1 0 0 0 1 1 1 0 0 0]
f = [1 0 0 0 0 0 0 1 0 0 0 0 0 0 0]
y = [1 0 0 0 0 0 0 1 0 0 0 0 0 0 0]


,Input
0,b_T[0]: outdoor
1,b_T[4]: outdoor
2,b_T[8]: ground
3,b_T[12]: outdoor
4,b_T[13]: outdoor
5,b_T[14]: outdoor
6,b_T[15]: setpoint
7,b_T[16]: outdoor
8,b_T[20]: outdoor
9,b_T[24]: ground


Node,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
Branch,,,,,,,,,,,,,,,
0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,-1,1,0,0,0,0,0,0,0,0,0,0,0,0
2,1,0,-1,0,0,0,0,0,0,0,0,0,0,0,0
3,1,0,-1,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
5,0,0,0,-1,1,0,0,0,0,0,0,0,0,0,0
6,1,0,0,0,-1,0,0,0,0,0,0,0,0,0,0
7,1,0,0,0,-1,0,0,0,0,0,0,0,0,0,0
8,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0


## 10. Conclusion

We defined a 54 m² two-zone pavilion with fixed geometry, materials, occupancy and boundary conditions. Its numbered thermal circuit includes air and building-mass temperatures and provides the DAE matrices and input pointers for the following notebooks.

**Sources:** [Climate.OneBuilding Switzerland weather files](https://climate.onebuilding.org/WMO_Region_6_Europe/CHE_Switzerland/index.html); [RIT DIRSIG thermodynamic parameter guidance](https://dirsig.cis.rit.edu/docs/new/thermal_cookbook.html); [ROCKWOOL RockFall datasheet](https://www.rockwool.com/siteassets/rw-uk/downloads/datasheets/rockfall-datasheet-oct-22.pdf); Christian Ghiaus, [03CubicBuilding.ipynb](https://github.com/cghiaus/dm4bem/blob/2eb8803cf038bd73c1618f2fd89771b9ba285ab4/03CubicBuilding.ipynb).
